<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week12/Day4/DailyChallenge/W12D4_Challenge_Preprocess_%26_fine_tune_transformer_based_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load data

In [8]:
# Import standard libraries for working with files and folders
import os
import zipfile

# Import a library for downloading files from the Internet
import requests

# Import pandas for loading and analyzing tabular data
import pandas as pd


# URL of the dataset archive
url = (
    "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/"
    "Week%206/W6D1%20GenAi%20France/"
    "Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip"
)

# Name of the downloaded ZIP file
zip_path = "bert_dataset.zip"

# Download the archive from GitHub
response = requests.get(url)
# Raise an error if the download failed
response.raise_for_status()

# Save the downloaded content to a local ZIP file
with open(zip_path, "wb") as file:
    file.write(response.content)

print("✅ Dataset downloaded successfully.")


# Folder where the archive will be extracted
extract_folder = "dataset"

# Open the ZIP archive and extract all files
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("✅ Archive extracted successfully.")


# Display the folder structure to see what files are available
for root, dirs, files in os.walk(extract_folder):
    for file in files:
        print(os.path.join(root, file))

✅ Dataset downloaded successfully.
✅ Archive extracted successfully.
dataset/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv
dataset/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip
dataset/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip


# Explore Data

In [9]:
# Display the updated folder structure after extraction
for root, dirs, files in os.walk(extract_folder):
    for file in files:
        print(os.path.join(root, file))

dataset/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv
dataset/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip
dataset/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip


In [10]:
# Import a library for working with compressed ZIP files
import zipfile

# Extract all nested ZIP archives (train.csv.zip and test.csv.zip)
for root, dirs, files in os.walk(extract_folder):
    for file in files:

        # Process only ZIP files
        if file.endswith(".zip"):

            # Full path to the ZIP archive
            zip_file_path = os.path.join(root, file)

            # Extract the archive into the same folder
            with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
                zip_ref.extractall(root)

            print(f"✅ Extracted: {file}")

✅ Extracted: test.csv.zip
✅ Extracted: train.csv.zip


In [11]:
# Build paths to the training and test datasets
train_path = os.path.join(
    extract_folder,
    "Basics of BERT and XLM-RoBERTa - PyTorch",
    "train.csv"
)

test_path = os.path.join(
    extract_folder,
    "Basics of BERT and XLM-RoBERTa - PyTorch",
    "test.csv"
)

# Load the CSV files into pandas DataFrames
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display dataset dimensions
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Display the first five rows
train_df.head()

Train shape: (12120, 6)
Test shape: (5195, 5)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [12]:
# Display label distribution
print(train_df["label"].value_counts().sort_index())

# Check for missing values
print(train_df.isnull().sum())

label
0    4176
1    3880
2    4064
Name: count, dtype: int64
id            0
premise       0
hypothesis    0
lang_abv      0
language      0
label         0
dtype: int64


# Understanding BERT and XLM-RoBERTa

## Initialize Tokenizer


In [13]:
# Import BERT and XLM-RoBERTa tokenizers
from transformers import BertTokenizer, XLMRobertaTokenizer

# Load the pretrained BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Load the pretrained XLM-RoBERTa tokenizer
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

## Learn about different pre-trained versions of these models and their characteristics.

In [14]:
# Display the vocabulary size of each tokenizer
print(f"BERT vocabulary size: {bert_tokenizer.vocab_size:,}")
print(f"XLM-RoBERTa vocabulary size: {xlmr_tokenizer.vocab_size:,}")

print("\n" + "=" * 60 + "\n")

# Display special tokens used by BERT
print("BERT special tokens:")
print(bert_tokenizer.special_tokens_map)

print("\n" + "=" * 60 + "\n")

# Display special tokens used by XLM-RoBERTa
print("XLM-RoBERTa special tokens:")
print(xlmr_tokenizer.special_tokens_map)

BERT vocabulary size: 30,522
XLM-RoBERTa vocabulary size: 250,002


BERT special tokens:
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


XLM-RoBERTa special tokens:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


## Tokenizing Text

In [15]:
# Select the first training example
premise = train_df.loc[0, "premise"]
hypothesis = train_df.loc[0, "hypothesis"]

# Display the original texts
print("Premise:")
print(premise)

print("\nHypothesis:")
print(hypothesis)

Premise:
and these comments were considered in formulating the interim rules.

Hypothesis:
The rules developed in the interim were put together with these comments in mind.


In [16]:
# Tokenize a pair of sentences using the BERT tokenizer
bert_encoding = bert_tokenizer(

    premise,
    hypothesis,

    # Add special tokens ([CLS], [SEP])
    add_special_tokens=True,

    # Return attention mask
    return_attention_mask=True,

    # Return PyTorch ("pt") tensors
    return_tensors="pt"
)

In [17]:
# Display all keys returned by the tokenizer
print("Returned keys:")
print(bert_encoding.keys())

Returned keys:
KeysView({'input_ids': tensor([[ 101, 1998, 2122, 7928, 2020, 2641, 1999, 5675, 3436, 1996, 9455, 3513,
         1012,  102, 1996, 3513, 2764, 1999, 1996, 9455, 2020, 2404, 2362, 2007,
         2122, 7928, 1999, 2568, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])})


In [18]:
# Display input token IDs
print("=" * 60)
print("Input IDs")
print("=" * 60)

print(bert_encoding["input_ids"])

Input IDs
tensor([[ 101, 1998, 2122, 7928, 2020, 2641, 1999, 5675, 3436, 1996, 9455, 3513,
         1012,  102, 1996, 3513, 2764, 1999, 1996, 9455, 2020, 2404, 2362, 2007,
         2122, 7928, 1999, 2568, 1012,  102]])


In [19]:
# Decode the token IDs back into text
decoded_text = bert_tokenizer.decode(
    bert_encoding["input_ids"][0]
)

# Display the decoded text
print(decoded_text)

[CLS] and these comments were considered in formulating the interim rules. [SEP] the rules developed in the interim were put together with these comments in mind. [SEP]


### token type IDs (=segment IDs)
BERT assignes sentence type (segment) to each token. Sentence A will have Type = 0, Sentence B - Type = 1

In [20]:
# Display token type IDs
print("Token type IDs:")
print(bert_encoding["token_type_ids"])

Token type IDs:
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])


In [21]:
# Display the attention mask
print("Attention mask:")
print(bert_encoding["attention_mask"])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])


# Preparing Input Data for the Model (Padding / Truncation)

In [22]:
# Display the shape of each returned tensor
for key, value in bert_encoding.items():
    print(f"{key}: {value.shape}")

input_ids: torch.Size([1, 30])
token_type_ids: torch.Size([1, 30])
attention_mask: torch.Size([1, 30])


In [23]:
# Tokenize the same sentence pair with padding and truncation
bert_encoding_padded = bert_tokenizer(

    premise,
    hypothesis,

    # Add special tokens ([CLS], [SEP])
    add_special_tokens=True,

    # Pad the sequence to the specified maximum length
    padding="max_length",

    # Truncate sequences longer than max_length
    truncation=True,

    # Set the maximum sequence length
    max_length=40,

    # Return attention mask
    return_attention_mask=True,

    # Return PyTorch tensors
    return_tensors="pt"
)

In [24]:
# Decode the padded sequence back into text
decoded_padded = bert_tokenizer.decode(
    bert_encoding_padded["input_ids"][0]
)

# Display the decoded sequence
print(decoded_padded)

[CLS] and these comments were considered in formulating the interim rules. [SEP] the rules developed in the interim were put together with these comments in mind. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [25]:
# Display the attention mask for the padded sequence
print(bert_encoding_padded["attention_mask"])

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [26]:
# Compare sequence lengths before and after padding
print(f"Original length: {bert_encoding['input_ids'].shape[1]}")
print(f"Padded length:   {bert_encoding_padded['input_ids'].shape[1]}")

Original length: 30
Padded length:   40


# Creating Cross-Validation Folds

In [27]:
from sklearn.model_selection import StratifiedKFold

# Create a Stratified K-Fold splitter
skf = StratifiedKFold(

    # Number of folds
    n_splits=5,

    # Shuffle the data before splitting
    shuffle=True,

    # Ensure reproducible results
    random_state=42
)

In [28]:
# Generate the train and validation indices for each fold
for fold, (train_idx, valid_idx) in enumerate(
    skf.split(train_df, train_df["label"])
):

    print(f"Fold {fold}")

    print(f"Train samples:      {len(train_idx)}")
    print(f"Validation samples: {len(valid_idx)}")

    print("-" * 40)

Fold 0
Train samples:      9696
Validation samples: 2424
----------------------------------------
Fold 1
Train samples:      9696
Validation samples: 2424
----------------------------------------
Fold 2
Train samples:      9696
Validation samples: 2424
----------------------------------------
Fold 3
Train samples:      9696
Validation samples: 2424
----------------------------------------
Fold 4
Train samples:      9696
Validation samples: 2424
----------------------------------------


In [29]:
# Display the first few indices from the first fold
for fold, (train_idx, valid_idx) in enumerate(
    skf.split(train_df, train_df["label"])
):

    print(f"Fold {fold}")

    print("First 10 train indices:")
    print(train_idx[:10])

    print()

    print("First 10 validation indices:")
    print(valid_idx[:10])

    break

Fold 0
First 10 train indices:
[ 0  1  2  4  5  7  8  9 10 11]

First 10 validation indices:
[ 3  6 21 26 27 28 30 32 33 35]


In [30]:
# Create train and validation DataFrames for the first fold
for fold, (train_idx, valid_idx) in enumerate(
    skf.split(train_df, train_df["label"])
):

    train_fold = train_df.iloc[train_idx]
    valid_fold = train_df.iloc[valid_idx]

    break

In [31]:
# Display the sizes of the first fold
print(train_fold.shape)
print(valid_fold.shape)

(9696, 6)
(2424, 6)


In [32]:
# Tokenize all training samples in the first fold
train_encoding = bert_tokenizer(

    train_fold["premise"].tolist(),
    train_fold["hypothesis"].tolist(),

    # Pad all sequences to the same length
    padding="max_length",

    # Truncate long sequences
    truncation=True,

    # Maximum sequence length
    max_length=128,

    # Return attention masks
    return_attention_mask=True,

    # Return PyTorch tensors
    return_tensors="pt"
)

In [33]:
# Display the keys returned by the tokenizer
print(train_encoding.keys())

KeysView({'input_ids': tensor([[  101,  1998,  2122,  ...,     0,     0,     0],
        [  101,  2122,  2024,  ...,     0,     0,     0],
        [  101,  4078, 20146,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  2590,  ...,     0,     0,     0],
        [  101,  2012,  1996,  ...,     0,     0,     0],
        [  101,  2005,  2370,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})


In [34]:
# Display tensor shapes
for key, value in train_encoding.items():
    print(f"{key}: {value.shape}")

input_ids: torch.Size([9696, 128])
token_type_ids: torch.Size([9696, 128])
attention_mask: torch.Size([9696, 128])


# Dataset

In [35]:
# Import the PyTorch Dataset base class
from torch.utils.data import Dataset
import torch


# Create a custom Dataset (inherited from PyTorch Dataset) for the NLI task
class NLIDataset(Dataset):

    def __init__(self, encodings, labels):

        # Store tokenized inputs
        self.encodings = encodings

        # Store labels
        self.labels = labels

    # Return the number of samples in the dataset
    def __len__(self):

        return len(self.labels)

    # Return a single sample from the dataset
    def __getitem__(self, idx):

        item = {}

        # Extract the tensors for the requested sample
        for key, value in self.encodings.items():
            item[key] = value[idx]

        # Return the corresponding label
        item["labels"] = torch.tensor(
            self.labels.iloc[idx],
            dtype=torch.long
        )

        # Return the completed sample
        return item

In [36]:
# Tokenize all validation samples in the first fold
valid_encoding = bert_tokenizer(

    valid_fold["premise"].tolist(),
    valid_fold["hypothesis"].tolist(),

    # Pad all sequences to the same length
    padding="max_length",

    # Truncate long sequences
    truncation=True,

    # Maximum sequence length
    max_length=128,

    # Return attention masks
    return_attention_mask=True,

    # Return PyTorch tensors
    return_tensors="pt"
)

In [37]:
train_dataset = NLIDataset(
    train_encoding,
    train_fold["label"]
)

valid_dataset = NLIDataset(
    valid_encoding,
    valid_fold["label"]
)

In [38]:
# Display dataset sizes
print(len(train_dataset))
print(len(valid_dataset))

9696
2424


In [39]:
# Display the first sample
sample = train_dataset[0]

print(sample.keys())

for key, value in sample.items():
    print(f"{key}: {value}")

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
input_ids: tensor([ 101, 1998, 2122, 7928, 2020, 2641, 1999, 5675, 3436, 1996, 9455, 3513,
        1012,  102, 1996, 3513, 2764, 1999, 1996, 9455, 2020, 2404, 2362, 2007,
        2122, 7928, 1999, 2568, 1012,  102,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0])
token_type_ids: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# Dataloader

In [40]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

In [41]:
# check dataloader
batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: {value.shape}")

input_ids: torch.Size([16, 128])
token_type_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16])


# Initialize model


In [42]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print(model.config.num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3


# Optimizer

In [43]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5 # learning rate
)

# Training loop

In [44]:
model.train()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## Forward pass №1

In [45]:
batch = next(iter(train_loader)) # take the 1st batch from the loader, train_loader is an object of DataLoader, that gives batches to a model

outputs = model(**batch)

print(outputs.loss)
print(outputs.logits.shape)

tensor(1.1485, grad_fn=<NllLossBackward0>)
torch.Size([16, 3])


## Fine-tuning

In [46]:
# Select the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

# Move the model to the selected device
model.to(device)

cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [47]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [48]:
from tqdm.auto import tqdm

# Number of training epochs
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    epoch_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:

        # Move the entire batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

        progress_bar.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}: Average Loss = {avg_loss:.4f}")

Epoch 1:   0%|          | 0/606 [00:00<?, ?it/s]

Epoch 1: Average Loss = 0.9946


Epoch 2:   0%|          | 0/606 [00:00<?, ?it/s]

Epoch 2: Average Loss = 0.7790


Epoch 3:   0%|          | 0/606 [00:00<?, ?it/s]

Epoch 3: Average Loss = 0.5877


In [49]:
print(next(model.parameters()).device)

cuda:0


# Model Evaluation

In [50]:
# Switch the model to evaluation mode
model.eval()

validation_loss = 0

# Disable gradient computation
with torch.no_grad():

    for batch in valid_loader:

        # Move the batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        validation_loss += outputs.loss.item()

avg_validation_loss = validation_loss / len(valid_loader)

print(f"Validation Loss: {avg_validation_loss:.4f}")

Validation Loss: 0.9190


In [52]:
# Import accuracy metric
from sklearn.metrics import accuracy_score

# Switch the model to evaluation mode
model.eval()

predictions = []
true_labels = []

# Disable gradient computation
with torch.no_grad():

    for batch in valid_loader:

        # Move the batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Forward pass
        outputs = model(**batch)

        # Predicted class
        preds = torch.argmax(outputs.logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batch["labels"].cpu().numpy())

# Compute accuracy
accuracy = accuracy_score(true_labels, predictions)

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.5982
